In [1]:
%%capture
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install requests pandas

In [2]:
import requests
import json
import pandas as pd
from tqdm.notebook import tqdm
import torch
from unsloth import FastLanguageModel
from trl import SFTTrainer
from transformers import TrainingArguments

# Replace with your HuggingFace Space URL
API_BASE_URL = "http://127.0.0.1:7860"


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [16]:
def collect_trajectories(num_episodes=5):
    data = []

    for ep in range(num_episodes):
        # Reset the environment (Task 2)
        session = requests.post(f"{API_BASE_URL}/reset", json={"task_id": 2, "seed": ep}).json()
        # Debugging: Print the session object
        print("Session object:", session)
        session_id = session["session_id"]
        obs = session["observation"]

        while True:
            # Here you would normally use a strong teacher model (e.g., GPT-4o or Claude)
            # to generate the 'expert' action for the SFT dataset.
            # For demonstration, we use a simple heuristic agent.

            text = (obs["subject"] + " " + obs["body"]).lower()
            if any(w in text for w in ["urgent", "critical", "outage", "rejecting requests", "broken", "crashing", "hanging", "code red", "failing silently", "500s"]): priority = "critical"
            else: priority = "medium"

            action_payload = {
                "assign_priority": priority,
                "assign_category": "technical",
                "response_text": "We are investigating the issue and will update you soon.",
                "escalate": priority == "critical"
            }

            # Record the state-action pair
            prompt = f"Ticket: {obs['subject']}\nBody: {obs['body']}\nTier: {obs['tier']}\nWhat is the correct priority, category, and response?"
            response = json.dumps({"priority": action_payload["assign_priority"], "category": action_payload["assign_category"], "response": action_payload["response_text"], "escalate": action_payload["escalate"]})

            data.append({
                "instruction": prompt,
                "output": response
            })

            # Step the environment
            step_res = requests.post(f"{API_BASE_URL}/step", json={"session_id": session_id, "action": action_payload}).json()
            # Debugging: Print step_res to see its contents
            print("Step response:", step_res)
            if "done" in step_res and step_res["done"]:
                break
            obs = step_res["observation"]

    return pd.DataFrame(data)

df = collect_trajectories(num_episodes=5)
print(f"Collected {len(df)} training examples.")
df.head()

Session object: {'session_id': 'ceae145a-f61b-49e4-98ae-57e39b5aa56f', 'observation': {'ticket_id': 'ATK-00001', 'active_policy_version': 'v1', 'world_state_summary': {'company_balance': 10000.0, 'churn_risk': 0.0, 'escalation_queue_size': 0, 'sla_breaches': 0, 'tickets_processed': 0, 'difficulty_level': 1, 'current_round': 1}, 'subject': "I'm really disappointed with Payment Module. The analytics", 'body': "I'm really disappointed with Payment Module. The analytics dashboard has been crashing intermittently and this affects my small business.", 'tier': 'Free'}, 'world_state': {'company_balance': 10000.0, 'churn_risk': 0.0, 'escalation_queue_size': 0, 'sla_breaches': 0, 'current_policy_version': 'v1', 'drift_events_fired': 0, 'agent_drift_accuracy': 0.0, 'stale_decisions_made': 0, 'hallucinations_caught': 0, 'difficulty_level': 1, 'tickets_processed': 0, 'multi_turn_active': False, 'catastrophic_failure': False, 'catastrophic_reason': '', 'current_round': 1, 'rounds_survived': 0, 'adap

,instruction,output
0,Ticket: I'm really disappointed with Payment M...,"{""priority"": ""critical"", ""category"": ""technica..."
1,Ticket: not urgent at all but the Analytics En...,"{""priority"": ""critical"", ""category"": ""technica..."
2,Ticket: CODE RED: I need to change my billing\...,"{""priority"": ""critical"", ""category"": ""technica..."
3,Ticket: CRITICAL ALERT: Our Business subscript...,"{""priority"": ""critical"", ""category"": ""technica..."
4,Ticket: not urgent at all but the Search Servi...,"{""priority"": ""critical"", ""category"": ""technica..."


In [10]:
max_seq_length = 2048
dtype = None # Auto detection
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 16,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = 16,
    lora_dropout = 0,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)

==((====))==  Unsloth 2026.4.8: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/457M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/270 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

tokenizer.json:   0%|          | 0.00/11.4M [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/605 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/614 [00:00<?, ?B/s]

unsloth/Qwen2.5-0.5B-Instruct-bnb-4bit does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


Unsloth 2026.4.8 patched 24 layers with 24 QKV layers, 24 O layers and 24 MLP layers.


In [12]:
alpaca_prompt = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert customer support agent.
{}

### Response:
{}
"""

def format_prompts(examples):
    instructions = examples["instruction"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        text = alpaca_prompt.format(instruction, output) + tokenizer.eos_token
        texts.append(text)
    return { "text" : texts }

from datasets import Dataset
dataset = Dataset.from_pandas(df)
dataset = dataset.map(format_prompts, batched = True)

Map:   0%|          | 0/17 [00:00<?, ? examples/s]

In [13]:
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/17 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 17 | Num Epochs = 20 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 8,798,208 of 502,830,976 (1.75% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Step,Training Loss
1,3.395293
2,3.409602
3,3.299399
4,3.003994
5,2.844417
6,2.390551
7,2.388898
8,2.155775
9,1.848575
10,1.545037


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.


In [19]:
FastLanguageModel.for_inference(model)

# Test on a new episode
session = requests.post(f"{API_BASE_URL}/reset", json={"task_id": 2, "seed": 42}).json()
session_id = session["session_id"]
obs = session["observation"]

prompt = f"Ticket: {obs['subject']}\nBody: {obs['body']}\nTier: {obs['tier']}\nWhat is the correct priority, category, and response?"
inputs = tokenizer([alpaca_prompt.format(prompt, "")], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 64, use_cache = True)
print(tokenizer.batch_decode(outputs, skip_special_tokens = True)[0])

Both `max_new_tokens` (=64) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/

Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.

### Instruction:
You are an expert customer support agent.
Ticket: URGENT: How do I set up multi-region failover
Body: URGENT: How do I set up multi-region failover on Storage Layer?
Tier: Team
What is the correct priority, category, and response?

### Response:

{"priority": "critical", "category": "technical", "response": "We are investigating the issue and will update you soon.", "escalate": true}



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
import torch
from environment import GauntletEnv
from generate_training_plots import plot_mean_reward, plot_per_step_reward, plot_difficulty
import json
import re

os.makedirs('results', exist_ok=True)

# Define heuristic agent
def heuristic_action(obs):
    text = (obs.get('subject', '') + ' ' + obs.get('body', '')).lower()
    priority = 'Critical' if any(w in text for w in ['urgent', 'critical', 'down', 'outage']) else 'Medium'
    return {
        'assign_priority': priority,
        'assign_category': 'Technical',
        'draft_response': 'We are looking into this.',
        'escalate': priority == 'Critical'
    }

FastLanguageModel.for_inference(model)
def llm_agent(obs):
    prompt = f"Ticket: {obs.get('subject', '')}\nBody: {obs.get('body', '')}\nTier: {obs.get('tier', '')}\nWhat is the correct priority, category, and response?"
    formatted_prompt = alpaca_prompt.format(prompt, "")
    inputs = tokenizer([formatted_prompt], return_tensors="pt").to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=64, use_cache=True, pad_token_id=tokenizer.eos_token_id)
    resp = tokenizer.batch_decode(outputs, skip_special_tokens=True)[0]
    
    try:
        json_str = re.search(r'\{.*?\}', resp.split('### Response:')[1], re.DOTALL).group(0)
        action = json.loads(json_str)
        return {
            'assign_priority': action.get('priority', 'Medium'),
            'assign_category': action.get('category', 'Technical'),
            'draft_response': action.get('response', ''),
            'escalate': action.get('escalate', False)
        }
    except:
        return heuristic_action(obs)

def run_eval_loop(num_episodes=30, agent_fn=heuristic_action, label="Agent", start_seed=42):
    episodes_data = []
    print(f"Running {label}...")
    
    env = GauntletEnv()
    for ep in tqdm(range(num_episodes), desc=label):
        obs = env.reset(task_id=2, attacker_enabled=True, drift_enabled=True, seed=start_seed + ep)
        done = False
        step_rewards = []
        difficulty_per_step = []
        while not done:
            action = agent_fn(obs)
            result = env.step(action)
            step_rewards.append(result['reward'])
            difficulty_per_step.append(env.world_state.difficulty_level)
            done = result['done']
            if not done:
                obs = result['observation']
                
        episodes_data.append({
            'step_rewards': step_rewards,
            'mean_reward': float(np.mean(step_rewards)) if step_rewards else 0.0,
            'difficulty_final': difficulty_per_step[-1] if difficulty_per_step else 1,
            'difficulty_per_step': difficulty_per_step
        })
    return episodes_data

EVAL_EPISODES = 30
baseline_eps = run_eval_loop(EVAL_EPISODES, heuristic_action, "Baseline")
trained_eps = run_eval_loop(EVAL_EPISODES, llm_agent, "Trained")

# Generate rich plots matching your results/ folder!
plot_mean_reward(baseline_eps, trained_eps)
plot_per_step_reward(baseline_eps, trained_eps)
plot_difficulty(baseline_eps, trained_eps)

b_means = [e['mean_reward'] for e in baseline_eps]
t_means = [e['mean_reward'] for e in trained_eps]
print(f"\nBaseline Mean: {np.mean(b_means):.3f}")
print(f"Trained Mean:  {np.mean(t_means):.3f}")
print("\nAll rich graphs have been saved to results/")